In [7]:
from __future__ import annotations

from dataclasses import dataclass
from bisect import bisect_left, bisect_right
from typing import Dict, List, Optional, Tuple, Any, Set

import numpy as np
import pandas as pd

# ---- encoding (your mapping) ----
ID_IDLE = -1
ID_SWITCHING_OFF = -2
ID_SWITCHING_ON = -3
ID_SLEEPING = -4

EPS = 1e-9


@dataclass(frozen=True)
class Interval:
    start: float
    end: float
    job_id: int
    state: str
    row_idx: int


def _is_nan(x) -> bool:
    return x is None or (isinstance(x, float) and np.isnan(x))


def _to_int_job_id(x) -> int:
    if _is_nan(x):
        return 0
    try:
        return int(float(x))
    except Exception:
        return int(x)


def _parse_nodes_cell(x) -> List[int]:
    if _is_nan(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    return [int(t) for t in s.split()]


def _parse_job_nodes_cell(x) -> List[int]:
    if _is_nan(x):
        return []
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(v) for v in x]
    s = str(x).strip()
    if not s:
        return []
    s = s.replace("[", "").replace("]", "").replace('"', "").replace("'", "")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return [int(float(p)) for p in parts]


def read_node_log(node_log_path: str) -> pd.DataFrame:
    df = pd.read_csv(node_log_path)

    # normalize names
    if "allocated_resources" in df.columns and "nodes" not in df.columns:
        df = df.rename(columns={"allocated_resources": "nodes"})
    if "starting_time" in df.columns and "start_time" not in df.columns:
        df = df.rename(columns={"starting_time": "start_time"})
    if "type" in df.columns and "state" not in df.columns:
        df = df.rename(columns={"type": "state"})

    required = {"start_time", "finish_time", "nodes", "state", "job_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"node_log.csv missing columns: {sorted(missing)}")

    df = df.copy()
    df["job_id"] = df["job_id"].apply(_to_int_job_id)
    df["start_time"] = df["start_time"].astype(float)
    df["finish_time"] = df["finish_time"].astype(float)
    df["nodes_parsed"] = df["nodes"].apply(_parse_nodes_cell)
    df["state"] = df["state"].astype(str)
    return df


def read_jobs(raw_job_log_path: str) -> pd.DataFrame:
    df = pd.read_csv(raw_job_log_path)
    required = {"job_id", "subtime", "start_time", "nodes"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"raw_job_log.csv missing columns: {sorted(missing)}")

    df = df.copy()
    df["job_id"] = df["job_id"].apply(_to_int_job_id)
    df["subtime"] = df["subtime"].astype(float)
    df["start_time"] = df["start_time"].astype(float)
    df["nodes_parsed"] = df["nodes"].apply(_parse_job_nodes_cell)

    # required size
    if "res" in df.columns:
        df["res"] = df["res"].apply(lambda x: int(float(x)) if not _is_nan(x) else 0)
    else:
        df["res"] = df["nodes_parsed"].apply(len)

    return df


def classify(itv: Interval) -> str:
    st = itv.state.lower()
    jid = itv.job_id
    if jid > 0:
        return "computing"
    if jid == ID_IDLE:
        return "idle"
    if jid == ID_SWITCHING_ON or "switching_on" in st:
        return "switching_on"
    if jid == ID_SWITCHING_OFF or "switching_off" in st:
        return "switching_off"
    if jid == ID_SLEEPING or "sleep" in st:
        return "sleeping"
    return "other"


def build_per_node(node_df: pd.DataFrame, num_nodes: int) -> Tuple[List[List[Interval]], List[List[float]]]:
    per_node: List[List[Interval]] = [[] for _ in range(num_nodes)]
    for idx, r in node_df.iterrows():
        nodes = r["nodes_parsed"]
        if not nodes:
            continue
        itv = Interval(
            start=float(r["start_time"]),
            end=float(r["finish_time"]),
            job_id=int(r["job_id"]),
            state=str(r["state"]),
            row_idx=int(idx),
        )
        for n in nodes:
            if 0 <= n < num_nodes:
                per_node[n].append(itv)

    starts: List[List[float]] = []
    for n in range(num_nodes):
        per_node[n].sort(key=lambda x: (x.start, x.end, x.row_idx))
        starts.append([itv.start for itv in per_node[n]])
    return per_node, starts


def interval_at(per_node: List[List[Interval]], starts: List[List[float]], node: int, t: float) -> Optional[Interval]:
    tl = per_node[node]
    if not tl:
        return None
    s = starts[node]
    i = bisect_right(s, t + EPS) - 1
    if i < 0:
        return None
    itv = tl[i]
    # [start, end)
    if itv.start - EPS <= t < itv.end - EPS:
        return itv
    if abs(t - itv.end) <= EPS and i + 1 < len(tl):
        itv2 = tl[i + 1]
        if itv2.start - EPS <= t < itv2.end - EPS:
            return itv2
    return None


def idle_count_at(per_node: List[List[Interval]], starts: List[List[float]], t: float, num_nodes: int) -> int:
    c = 0
    for n in range(num_nodes):
        itv = interval_at(per_node, starts, n, t)
        if itv is not None and classify(itv) == "idle":
            c += 1
    return c


def next_compute_start(per_node: List[List[Interval]], node: int, idx: int) -> Optional[float]:
    tl = per_node[node]
    for k in range(idx + 1, len(tl)):
        if classify(tl[k]) == "computing":
            return tl[k].start
    return None


# ---------------- Scheduler checks ----------------

def check_switch_on_wasted(per_node: List[List[Interval]]) -> pd.DataFrame:
    """
    switching_on ends at t, but next state at t isn't computing.
    Reports delay until next compute (if ever).
    """
    rows = []
    for node, tl in enumerate(per_node):
        for i, itv in enumerate(tl):
            if classify(itv) != "switching_on":
                continue
            t_end = itv.end

            # find interval that starts exactly at t_end (contiguous)
            nxt_i = None
            for j in range(i + 1, len(tl)):
                if abs(tl[j].start - t_end) <= 1e-6:
                    nxt_i = j
                    break
                if tl[j].start > t_end + 1e-6:
                    break

            if nxt_i is None:
                continue

            nxt = tl[nxt_i]
            if classify(nxt) == "computing":
                continue

            t_comp = next_compute_start(per_node, node, nxt_i)
            rows.append({
                "node": node,
                "switch_on_start": itv.start,
                "switch_on_end": itv.end,
                "next_kind": classify(nxt),
                "delay_to_next_compute": (None if t_comp is None else (t_comp - t_end)),
                "switch_on_row": itv.row_idx,
            })

    df = pd.DataFrame(rows, columns=[
        "node","switch_on_start","switch_on_end","next_kind","delay_to_next_compute","switch_on_row"
    ])
    if not df.empty:
        df = df.sort_values(["switch_on_end","node"]).reset_index(drop=True)
    return df


def check_job_arrival_not_started_despite_idle_capacity(
    jobs_df: pd.DataFrame,
    per_node: List[List[Interval]],
    starts: List[List[float]],
    num_nodes: int,
) -> pd.DataFrame:
    """
    Job waits (start_time > subtime) even though idle nodes at arrival >= res.
    Assumes work-conserving policy.
    """
    rows = []
    for _, r in jobs_df.iterrows():
        jid = int(r["job_id"])
        sub = float(r["subtime"])
        st = float(r["start_time"])
        res = int(r["res"])
        wait = st - sub
        if wait <= EPS:
            continue
        idle = idle_count_at(per_node, starts, sub + EPS, num_nodes)
        if idle >= res:
            rows.append({
                "job_id": jid,
                "subtime": sub,
                "start_time": st,
                "waiting": wait,
                "res": res,
                "idle_nodes_at_arrival": idle,
            })

    df = pd.DataFrame(rows, columns=[
        "job_id","subtime","start_time","waiting","res","idle_nodes_at_arrival"
    ])
    if not df.empty:
        df = df.sort_values(["waiting","subtime"], ascending=[False, True]).reset_index(drop=True)
    return df


def check_premature_switch_off_causing_delay(
    jobs_df: pd.DataFrame,
    per_node: List[List[Interval]],
    starts: List[List[float]],
    num_nodes: int,
    lookahead: float = 60.0,
) -> pd.DataFrame:
    """
    Your case:
    node n is IDLE right before switching_off starts at t_off,
    but there exists some already-arrived waiting job that would become feasible soon
    if n stayed idle (i.e., within lookahead), while without n it is not feasible.

    Implementation:
    - sample feasibility only at event times (node state boundaries + job times).
    - use a counterfactual: "idle_count + 1" (keeping this node idle).
    - only flag if this node is the *marginal* node: idle < res and idle+1 >= res at some t_candidate.
    """
    # event times (where idle count can change)
    times: Set[float] = set()
    for tl in per_node:
        for itv in tl:
            times.add(itv.start)
            times.add(itv.end)
    times |= set(jobs_df["subtime"].tolist())
    times |= set(jobs_df["start_time"].tolist())
    times_sorted = sorted(times)

    # precompute idle_count at those times (slightly after boundary)
    idle_cache: Dict[float, int] = {}
    for t in times_sorted:
        idle_cache[t] = idle_count_at(per_node, starts, t + EPS, num_nodes)

    def window_times(t0: float, t1: float) -> List[float]:
        a = bisect_left(times_sorted, t0 - 1e-9)
        b = bisect_right(times_sorted, t1 + 1e-9)
        return times_sorted[a:b]

    rows = []

    for node, tl in enumerate(per_node):
        if not tl:
            continue

        for i, itv in enumerate(tl):
            if classify(itv) != "switching_off":
                continue
            t_off = itv.start
            t_off_end = itv.end

            # only consider "scheduler chose to switch off an IDLE node"
            prev = interval_at(per_node, starts, node, t_off - 1e-6)
            if prev is None or classify(prev) != "idle":
                continue

            waiting = jobs_df[(jobs_df["subtime"] <= t_off + EPS) & (jobs_df["start_time"] > t_off + EPS)]
            if waiting.empty:
                continue

            best = None  # (avoidable_delay, rowdict)
            for _, jr in waiting.iterrows():
                jid = int(jr["job_id"])
                sub = float(jr["subtime"])
                actual_start = float(jr["start_time"])
                res = int(jr["res"])

                t_end = min(t_off + lookahead, actual_start)
                if t_end <= t_off + EPS:
                    continue

                cand = None
                for t in window_times(t_off, t_end):
                    idle = idle_cache[t]
                    if idle < res and (idle + 1) >= res:
                        cand = t
                        break

                if cand is None:
                    continue

                avoid = actual_start - cand
                if avoid <= EPS:
                    continue

                # estimate transition waste relevant to this node before job starts
                on_dur = None
                # find first switching_on after this switch_off, before job starts
                for j in range(i + 1, len(tl)):
                    if tl[j].start < t_off_end - EPS:
                        continue
                    if tl[j].start > actual_start + EPS:
                        break
                    if classify(tl[j]) == "switching_on":
                        on_dur = tl[j].end - tl[j].start
                        break

                off_dur = t_off_end - t_off
                total_transition = off_dur + (on_dur if on_dur is not None else 0.0)

                node_in_actual_alloc = (node in set(jr["nodes_parsed"]))

                row = {
                    "node": node,
                    "switch_off_start": t_off,
                    "switch_off_end": t_off_end,
                    "job_id": jid,
                    "job_subtime": sub,
                    "job_res": res,
                    "candidate_start_if_not_switched_off": cand,
                    "actual_start": actual_start,
                    "avoidable_delay": avoid,
                    "idle_at_candidate_actual": idle_cache[cand],
                    "off_duration": off_dur,
                    "on_duration_before_job_start": on_dur,
                    "transition_time_sum_before_job_start": total_transition,
                    "node_in_actual_job_allocation": node_in_actual_alloc,
                    "switch_off_row": itv.row_idx,
                }

                if best is None or avoid > best[0]:
                    best = (avoid, row)

            if best is not None:
                rows.append(best[1])

    df = pd.DataFrame(rows, columns=[
        "node","switch_off_start","switch_off_end","job_id","job_subtime","job_res",
        "candidate_start_if_not_switched_off","actual_start","avoidable_delay",
        "idle_at_candidate_actual","off_duration","on_duration_before_job_start",
        "transition_time_sum_before_job_start","node_in_actual_job_allocation","switch_off_row"
    ])
    if not df.empty:
        df = df.sort_values(["avoidable_delay","switch_off_start"], ascending=[False, True]).reset_index(drop=True)
    return df


def run_scheduler_logic_checker(
    node_log_path: str,
    raw_job_log_path: str,
    num_nodes: int = 128,
    lookahead: float = 60.0,
) -> Dict[str, pd.DataFrame]:
    node_df = read_node_log(node_log_path)
    jobs_df = read_jobs(raw_job_log_path)
    per_node, starts = build_per_node(node_df, num_nodes=num_nodes)

    return {
        "switch_on_wasted": check_switch_on_wasted(per_node),
        "arrival_wait_despite_idle_capacity": check_job_arrival_not_started_despite_idle_capacity(
            jobs_df, per_node, starts, num_nodes=num_nodes
        ),
        "premature_switch_off_causing_delay": check_premature_switch_off_causing_delay(
            jobs_df, per_node, starts, num_nodes=num_nodes, lookahead=lookahead
        ),
    }


if __name__ == "__main__":
    report = run_scheduler_logic_checker(
        node_log_path="EASY_V3/node_log.csv",
        raw_job_log_path="EASY_V3/raw_job_log.csv",
        num_nodes=128,
        lookahead=60.0,   # "close future" window; set 10/30/300 depending on what you mean
    )

    for name, df in report.items():
        print(f"{name}: {len(df)}")
        if len(df):
            df.to_csv(f"sched_{name}.csv", index=False)
    print("Wrote sched_*.csv for non-empty results.")


switch_on_wasted: 278
arrival_wait_despite_idle_capacity: 102
premature_switch_off_causing_delay: 104
Wrote sched_*.csv for non-empty results.
